# PWR MOXUO2

### This is a computation of the oecd pwr mox-UO2 initial core ![](./pwr_moxuo2.png)

In [ ]:
import inspect

import numpy as np
from IPython.display import Code
from numba.np.unsafe.ndarray import *  # noqa: F403

from dorban.examples import oecd_pwr_mox_uo2
from dorban.solve_equation import solve_k

## The problem
The cross sections read from the benchmark's files, the materials, the geometry, the arrangement of the materials in the core (both in the default composition, all rods outside, and the ARI composition, all rods inside) and the cores are defined in `dorban.examples.oecd_pwr_mox_uo2`, which the benchmark tests use too:

In [ ]:
Code(inspect.getsource(oecd_pwr_mox_uo2), language="python")

## Define the cores of both configurations

In [ ]:
system, ari_system = oecd_pwr_mox_uo2.cores()

## Define the splitting parameters

In [ ]:
settings = oecd_pwr_mox_uo2.fd_settings(system)

## Solve the diffusion equation

In [ ]:
k, flux = solve_k(system, settings)
ari_k,ari_flux = solve_k(ari_system,settings)

### The reference for k with all rods outside is 1.06379

In [ ]:
print(f"k = {k:.5f}, reference {oecd_pwr_mox_uo2.REFERENCE_K}")

### The reference control worth is 6850 pcm

In [ ]:
print(f"control rod worth = {abs(1/k-1/ari_k)*1e5:.0f} pcm, reference {oecd_pwr_mox_uo2.REFERENCE_ROD_WORTH*1e5:.0f} pcm")

## Compare the power to the one computed by PARCS and the one computed by the transport code Decart

In [ ]:
reference_power = oecd_pwr_mox_uo2.REFERENCE_POWER
reference_ari = oecd_pwr_mox_uo2.REFERENCE_POWER_ARI
transport_power = oecd_pwr_mox_uo2.TRANSPORT_POWER

In [ ]:
power, ari_power = oecd_pwr_mox_uo2.assembly_powers(system, flux, ari_system, ari_flux)

In [ ]:
difference = np.abs(reference_power-power)/reference_power
transport_dif = np.abs(np.array(reference_power)-np.array(transport_power))/np.array(transport_power)
np.linalg.norm(transport_dif)/np.sqrt(len(transport_dif))
adiffernece = np.abs(reference_ari-ari_power)/reference_ari

In [ ]:
np.max(transport_dif)

In [ ]:
print(f"largest relative difference of the assembly powers from the reference, rods out: {np.max(difference):.2%}")

In [ ]:
print(f"largest relative difference of the assembly powers from the reference, rods in: {np.max(adiffernece):.2%}")

## Compare NEM with finite differences

In [ ]:
from dorban.nem.solve_nem import NEMSettings

aro_nem = oecd_pwr_mox_uo2.nem_core(system)
ari_nem = oecd_pwr_mox_uo2.nem_core(ari_system)

In [ ]:
aro_k_nem,aro_flux_nem=solve_k(aro_nem,NEMSettings(split=aro_nem.geometry.uniform_split(3)))

In [ ]:
ari_k_nem,ari_flux_nem=solve_k(ari_nem,NEMSettings(split=ari_nem.geometry.uniform_split(3)))

In [ ]:
print(f"rods out: NEM k = {aro_k_nem:.5f}, finite differences k = {k:.5f}")

In [ ]:
print(f"control rod worth: NEM {abs(1/aro_k_nem-1/ari_k_nem)*1e5:.0f} pcm, "
      f"finite differences {abs(1/k-1/ari_k)*1e5:.0f} pcm")

In [ ]:
print(f"rods out: largest relative difference of the flux, NEM against finite differences: {max(np.abs((aro_flux_nem-flux)/flux)):.2%}")

In [ ]:
print(f"rods in: largest relative difference of the flux, NEM against finite differences: {max(np.abs((ari_flux_nem-ari_flux)/ari_flux)):.2%}")